In [1]:
# 1️⃣ Import Libraries
import json
import random
from copy import deepcopy

# 2️⃣ Load Dataset
TRAIN_PATH = "/kaggle/input/datasets/abdullahshheikh/pii-masking-data/data.json"
TEST_PATH = "/kaggle/input/datasets/abdullahshheikh/pii-masking-data/test_data.json"

with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    train_data = json.load(f)

with open(TEST_PATH, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print("Train samples:", len(train_data))
print("Test samples:", len(test_data))

# 3️⃣ Extract Names from Dataset
def extract_names(dataset):
    first_names = set()
    last_names = set()
    for sample in dataset:
        tokens = sample["tokens"]
        tags = sample["ner_tags"]
        current_name = []
        for token, tag in zip(tokens, tags):
            if tag == "B-PER":
                current_name = [token]
            elif tag == "I-PER":
                current_name.append(token)
            else:
                if current_name:
                    if len(current_name) >= 1:
                        first_names.add(current_name[0].lower())
                    if len(current_name) >= 2:
                        last_names.add(current_name[-1].lower())
                    current_name = []
        # Check at end of sequence
        if current_name:
            if len(current_name) >= 1:
                first_names.add(current_name[0].lower())
            if len(current_name) >= 2:
                last_names.add(current_name[-1].lower())
    return list(first_names), list(last_names)

first_names, last_names = extract_names(train_data)
print("Unique first names:", len(first_names))
print("Unique last names:", len(last_names))

# 4️⃣ Email Generator (Realistic)
domains = ["gmail.com", "yahoo.com", "outlook.com", "hotmail.com"]

def generate_email(f_name=None, l_name=None):
    # If names aren't provided, pick random ones from the global list
    first = f_name if f_name else random.choice(first_names)
    last = l_name if l_name else random.choice(last_names)
    domain = random.choice(domains)
    
    patterns = [
        f"{first}.{last}@{domain}",
        f"{first}{last}@{domain}",
        f"{first}_{last}@{domain}",
        f"{first}{random.randint(10,99)}@{domain}"
    ]
    return random.choice(patterns).lower()

# 5️⃣ Add Email Token in Sequence
def insert_email_in_sequence(sample, insert_prob=0.3):
    tokens = deepcopy(sample["tokens"])
    tags = deepcopy(sample["ner_tags"])
    
    # 1. Identify all PER spans (to extract the actual name in the sentence)
    per_spans = []
    current_span = []
    for i, tag in enumerate(tags):
        if tag == "B-PER":
            current_span = [i]
        elif tag == "I-PER" and current_span:
            current_span.append(i)
        else:
            if current_span:
                per_spans.append(current_span)
                current_span = []
    
    if not per_spans or random.random() > insert_prob:
        return sample
    
    # 2. Pick a random name span to associate with an email
    target_span = random.choice(per_spans)
    
    # 3. 70/30 Rule: Should the email match the name in the sentence?
    if random.random() < 0.7:
        # Use names from the actual sequence
        f_name = tokens[target_span[0]]
        l_name = tokens[target_span[-1]] if len(target_span) > 1 else None
        email_val = generate_email(f_name, l_name)
    else:
        # Use random names from the pool
        email_val = generate_email()
    
    # 4. Insertion point: After the last token of the name span
    insert_at = target_span[-1] + 1
    
    # Optional: Add a connector like "(" or "at"
    connector = random.choice(["(", "at", "email:", ""])
    if connector:
        tokens.insert(insert_at, connector)
        tags.insert(insert_at, "O")
        insert_at += 1
    
    tokens.insert(insert_at, email_val)
    tags.insert(insert_at, "B-EMAIL")
    
    if connector == "(":
        tokens.insert(insert_at + 1, ")")
        tags.insert(insert_at + 1, "O")

    return {
        "tokens": tokens,
        "ner_tags": tags,
        "lang": sample.get("lang", "en"),
        "sequence": " ".join(tokens)
    }

# 6️⃣ Apply to Dataset (~40% of samples will get emails)
train_data_augmented = [insert_email_in_sequence(s, insert_prob=0.4) for s in train_data]
test_data_augmented = [insert_email_in_sequence(s, insert_prob=0.4) for s in test_data]

print("Original train size:", len(train_data))
print("Augmented train size:", len(train_data_augmented))

# 7️⃣ Validate Dataset Tags
valid_tags = {"O", "B-PER", "I-PER", "B-EMAIL", "I-EMAIL"}

def validate_dataset(dataset):
    for sample in dataset:
        if len(sample["tokens"]) != len(sample["ner_tags"]):
            raise ValueError("Length mismatch in sample:", sample)
        for tag in sample["ner_tags"]:
            if tag not in valid_tags:
                raise ValueError("Invalid tag found:", tag)
    print("Dataset validation passed.")

validate_dataset(train_data_augmented)

# 8️⃣ Save Processed Dataset
OUTPUT_TRAIN = "/kaggle/working//train_processed.json"
OUTPUT_TEST = "/kaggle/working/test_processed.json"


with open(OUTPUT_TRAIN, "w", encoding="utf-8") as f:
    json.dump(train_data_augmented, f, indent=2)

with open(OUTPUT_TEST, "w", encoding="utf-8") as f:
    json.dump(test_data_augmented, f, indent=2)

print("Processed datasets saved.")

# 9️⃣ Analysis: Print 20 samples where emails were added
print("\n--- ANALYZING AUGMENTED DATA (20 SAMPLES) ---")
count = 0
for i, sample in enumerate(train_data_augmented):
    if "B-EMAIL" in sample["ner_tags"]:
        count += 1
        
        # Identify the email and the person for visual check
        email_idx = sample["ner_tags"].index("B-EMAIL")
        email_token = sample["tokens"][email_idx]
        
        # Just find the first name in the sequence to check matching
        person_tokens = [t for t, tag in zip(sample["tokens"], sample["ner_tags"]) if tag in ["B-PER", "I-PER"]]
        
        print(f"Sample #{count} (Index {i}):")
        print(f"  Names Found: {person_tokens[:4]}") # Print first few name tokens
        print(f"  Email Added: {email_token}")
        print(f"  Full Sequence: {sample['sequence']}\n")
        
    if count == 20:
        break

Train samples: 28516
Test samples: 3650
Unique first names: 9830
Unique last names: 12483
Original train size: 28516
Augmented train size: 28516
Dataset validation passed.
Processed datasets saved.

--- ANALYZING AUGMENTED DATA (20 SAMPLES) ---
Sample #1 (Index 12):
  Names Found: ['Joss', 'Whedon']
  Email Added: josswhedon@yahoo.com
  Full Sequence: Joss Whedon at josswhedon@yahoo.com was credited as executive producer throughout the run of the series , and for the first five seasons ( 1997 – 2001 ) he was also the showrunner , supervising the writing and all aspects of production .

Sample #2 (Index 15):
  Names Found: ['Syverson', 'Yagelski']
  Email Added: yagelskiwaltz@hotmail.com
  Full Sequence: While a primary concern has been the relationship between the writing process and natural places , concepts of spatiality also apply to cyberspace and online writing - in MUDs , MOOs , Internet Relay Chat , Instant Messages , and e-mail ( Syverson , 1999 ; Yagelski at yagelskiwaltz@hotm